[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Parameters


## What you will be able to do

Put every value into a statement through a placeholder, by position with `?` or by name with
`:station`, including text full of apostrophes, `None`, and a list of values for `IN`. Choose a
table, a column or a sort order safely when it has to vary, which no placeholder can do. Recognize a
query built with an f-string, and show what a stranger's text does to it.


## The idea

### The problem

Every query in this guide so far has taken its values through a `?`. An f-string looks simpler:
`f"SELECT day, note FROM notes WHERE station = '{station}'"` reads exactly like the query it builds,
and it works the first time it is tried, with `Oslo`. Then the values stop being so tidy. A note that
says `The gauge's lid blew off` goes into the same kind of f-string, and its apostrophe ends the SQL
text early, so the insert fails with a syntax error. A search box hands the f-string whatever someone
typed, and someone types `' OR '1'='1`. That text closes the quote the f-string opened and adds a
condition that is true for every row, so the search returns every note in the file, with no error.
A little more typing returns the definition of every table instead.

That second failure is **SQL injection**: text that arrives as a value, but ends up as part of the
SQL. It cannot happen when the value never becomes SQL text, which is what a placeholder guarantees.

### What a parameter is

> A **parameter** is a value handed to `execute` beside the SQL, never written into it. The SQL
> marks the place for every value with a **placeholder**: a `?`, filled in order from a tuple or a
> list, or a name after a colon, such as `:station`, filled from a dictionary by name. SQLite
> prepares the statement with its placeholders still empty, and then **binds** a value into each
> one, so a parameter can only ever be a value: an apostrophe in it is a character, and text that
> looks like SQL is text. A placeholder can stand wherever a value can, and nowhere else, so never
> for a table name, a column name, or a word such as `DESC`.

### Why it works that way

- **The statement is prepared before any value is bound.** By the time a parameter arrives, SQLite
  has already decided what the statement does, and nothing in the value can change that.
- **An f-string builds SQL text, and SQL text is code.** Python does not know that the braces sit
  inside quotes in SQL, so a quote in the value ends the SQL string, and SQLite reads whatever
  follows as SQL.
- **A parameter keeps its type.** `None`, `int`, `float`, `str` and `bytes` arrive as SQLite's
  `NULL`, integer, real, text and blob, with no text in between, so `None` is never the word `None`.
- **Repeated SQL text is not prepared again.** sqlite3 keeps up to 128 prepared statements for
  every connection, the `cached_statements` argument of `connect`, and finds them by their text. A
  query with placeholders has one text whatever its values, while an f-string makes a new text,
  prepared afresh, for every value it is given.
- **Names are part of the statement.** SQLite cannot prepare a statement without knowing its tables
  and columns, so those must be in the SQL text, chosen by the program from names it already knows.
- **A sequence fills question marks, and a dictionary fills names.** A statement uses one style or
  the other, and Python 3.14 refuses a tuple for named placeholders, where earlier versions only
  warned.

### Where you will meet this

Every Python database driver takes its values beside the SQL, as PEP 249 sets out, though the
placeholder changes. psycopg, in the **asyncpg and psycopg3, Deep Dive** guide, writes `%s` and
`%(name)s`, which look like Python's own `%` formatting and are not: psycopg fills them itself, and
its documentation says never to merge values into a query yourself. asyncpg writes `$1`, `execute` in
the **DuckDB, Deep Dive** guide takes `?`, `$1` and `$name`, and `text()` in the **SQLAlchemy, Deep
Dive** guide takes `:name`, as sqlite3 does. The limit is the same in all of them: a placeholder
stands for a value, so a table or column name that has to vary is chosen from a fixed set, or, in
psycopg, quoted with its `sql.Identifier`.

### What this notebook covers

- A value in its place: `?` and a tuple
- Text that SQL would misread: apostrophes, semicolons and `--`
- Types, and `None`, including `IS ?` for a value that may be missing
- Named placeholders, filled from a dictionary, one row at a time or with `executemany`
- A list of values for `IN`, with question marks counted from its length
- Names that cannot be parameters: a sort order chosen from a fixed set
- When to write question marks, when names, and when an f-string
- A search whose filters are all optional, built from fixed pieces of SQL
- Seven errors: a string where a tuple belongs, a table name as a parameter, a dictionary missing a
  name, a list for `IN`, a tuple for named placeholders, a note with an apostrophe in an f-string,
  and a search that an f-string lets anyone rewrite

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE notes (station TEXT, day TEXT, note TEXT)")
conn.executemany("INSERT INTO notes VALUES (?, ?, ?)", [
    ("Svalbard", "2025-03-02", "The sensor's heater failed"),
    ("Oslo", "2025-07-14", "Mast repainted"),
])

search = "SELECT day, note FROM notes WHERE station = ?"
for typed in ["Svalbard", "Tromso", "' OR '1'='1"]:
    print(f"{typed!r:14} {conn.execute(search, (typed,)).fetchall()}")
conn.close()
```

```
'Svalbard'     [('2025-03-02', "The sensor's heater failed")]
'Tromso'       []
"' OR '1'='1"  []
```

One search, run with three things a person might type. The note's apostrophe went in as a character
and came back as one, and the last search, written to rewrite the query, was only ever compared with
station names, and matched none. Pasted into the SQL with an f-string instead, the note would have
been a syntax error, and the last search would have returned every note, which Common errors shows.


## Setup

Six imports, the year of readings, and a database built from it: the two tables the **Tables and
Queries** notebook designed, and a third for notes that people write about the stations.

- `sqlite3` builds the database and runs every statement
- `warnings` catches the warning that Python 3.12 and 3.13 give in one of the Common errors
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the database in it
- `shutil` removes the scratch folder at the end

The `notes` table starts empty, and the worked examples fill it.


In [1]:
import math
import shutil
import sqlite3
import warnings
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE notes (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                        day TEXT NOT NULL, note TEXT NOT NULL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()

print("built", DATABASE)


built scratch/stations.db


## Worked examples

### A value in its place

A `?` in the SQL marks the place for a value, and the tuple passed after the SQL fills the marks in
order. The SQL text stays the same whatever the values are, so one statement answers the same
question for any station and any temperature:


In [2]:
conn = sqlite3.connect(DATABASE)
station_ids = dict(conn.execute("SELECT name, id FROM stations"))

hours_below = """
    SELECT COUNT(*)
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = ? AND r.celsius < ?
"""
for station, celsius in [("Oslo", -5), ("Tromso", -5), ("Svalbard", -15)]:
    hours = conn.execute(hours_below, (station, celsius)).fetchone()[0]
    print(f"{station:<9} below {celsius:>3}: {hours:>4} hours")

print("Svalbard's latitude:", conn.execute("SELECT latitude FROM stations WHERE name = ?", ("Svalbard",)).fetchone()[0])


Oslo      below  -5:  134 hours
Tromso    below  -5: 1041 hours
Svalbard  below -15:  411 hours
Svalbard's latitude: 78.22


The first `?` took the station's name and the second the temperature, in the order the tuple holds
them. A single value still goes in a tuple, and a tuple of one needs its comma, as in
`("Svalbard",)`: without the comma the parentheses only group, which the Common error
`Incorrect number of bindings supplied` shows. `station_ids` came from a query that returns pairs, a
name and an id, which `dict` turns straight into a dictionary.

### Text that SQL would misread

A value that goes in through a placeholder is never read as SQL, so it needs no escaping. Notes that
people write about the stations hold apostrophes, semicolons and double dashes, which all mean
something in SQL text, and all go in and come back exactly as written:


In [3]:
notes = [
    ("Svalbard", "2025-03-02", "The sensor's heater failed, so the station sent nothing all day"),
    ("Oslo", "2025-07-14", "Mast repainted; readings from 09:00 to 11:00 may run warm"),
    ("Tromso", "2025-10-30", 'A visitor asked "is it -- really -- always this cold?"'),
]
for station, day, note in notes:
    conn.execute("INSERT INTO notes (station_id, day, note) VALUES (?, ?, ?)", (station_ids[station], day, note))
conn.commit()

for day, note in conn.execute("SELECT day, note FROM notes ORDER BY day"):
    print(day, note)


2025-03-02 The sensor's heater failed, so the station sent nothing all day
2025-07-14 Mast repainted; readings from 09:00 to 11:00 may run warm
2025-10-30 A visitor asked "is it -- really -- always this cold?"


Pasted into SQL between quotes, the apostrophe in `sensor's` would end the text early, and SQLite
would read whatever followed it as SQL, where a semicolon ends a statement and `--` starts a comment.
As parameters they are characters like any other, which is also why a stranger's text cannot change
a statement through a placeholder.

### Types, and None

A parameter keeps its Python type on the way in: `int`, `float`, `str` and `bytes` become SQLite's
integer, real, text and blob, and `None` becomes `NULL`. SQLite's `typeof` reports what arrived:


In [4]:
for value in [42, -17.3, "Svalbard", b"\x00\xff", None, True]:
    kind = conn.execute("SELECT typeof(?)", (value,)).fetchone()[0]
    print(f"{value!r:<12} {kind}")


42           integer
-17.3        real
'Svalbard'   text
b'\x00\xff'  blob
None         null
True         integer


`True` arrived as the integer 1, since a `bool` is an `int` in Python. Other types, such as a
`Decimal`, need an adapter first, which the **Adapters and Converters** notebook covers.

`NULL` needs care in a comparison. `= ?` given `None` matches nothing, just as `= NULL` did in the
**Tables and Queries** notebook, while `IS ?` matches `NULL` to `NULL` and any other value to an
equal one, so a query that may be handed `None` compares with `IS`:


In [5]:
for celsius in [None, -17.3]:
    equal = conn.execute("SELECT COUNT(*) FROM readings WHERE celsius = ?", (celsius,)).fetchone()[0]
    same = conn.execute("SELECT COUNT(*) FROM readings WHERE celsius IS ?", (celsius,)).fetchone()[0]
    print(f"{celsius!r:<6} = ? found {equal:>2}, IS ? found {same:>2}")


None   = ? found  0, IS ? found 24
-17.3  = ? found  1, IS ? found  1


### Named placeholders

A placeholder can have a name, written after a colon, as in `:station`, and a dictionary fills it by
name. The order of the dictionary's keys no longer matters, a name used twice in the SQL takes its
value once, and keys the statement does not use are ignored:


In [6]:
day_range = """
    SELECT COUNT(r.celsius), MIN(r.celsius), MAX(r.celsius)
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = :station AND r.hour >= :day AND r.hour < date(:day, '+1 day')
"""
print(conn.execute(day_range, {"station": "Svalbard", "day": "2025-03-01"}).fetchone())
print(conn.execute(day_range, {"day": "2025-03-02", "station": "Svalbard", "asked_by": "a visitor"}).fetchone())


(24, -14.7, -8.0)
(0, None, None)


`:day` appears twice, as the start of the day and, through `date(:day, '+1 day')`, as the start of
the next, and each dictionary gave it once. The second dictionary listed its keys in another order
and carried a key the statement never asked for, and neither mattered. It asked about 2 March, when
Svalbard sent no readings, so the count is 0, with no lowest or highest.

Rows that arrive as dictionaries, from a form or from JSON, go into `executemany` as they are, with
whatever extra keys they carry:


In [7]:
incoming = [
    {"station_id": station_ids["Bergen"], "day": "2025-11-03", "note": "Rain gauge cleared of leaves",
     "form": "maintenance"},
    {"day": "2025-12-01", "note": "Heater checked before winter; it's working",
     "station_id": station_ids["Kirkenes"]},
]
conn.executemany("INSERT INTO notes (station_id, day, note) VALUES (:station_id, :day, :note)", incoming)
conn.commit()

print("notes:", conn.execute("SELECT COUNT(*) FROM notes").fetchone()[0])


notes: 5


### A list of values for IN

A placeholder holds one value, so `IN` needs a `?` for every value in its list. How many values
there are is known only when the program runs, so the program writes the marks from the list's
length: `"?" * 3` is `???`, and `", ".join` puts a comma and a space between its characters. The
values themselves still go in as parameters:


In [8]:
wanted = ["Tromso", "Svalbard", "Kirkenes"]
marks = ", ".join("?" * len(wanted))
query = f"SELECT name, latitude FROM stations WHERE name IN ({marks}) ORDER BY latitude DESC"

print(query)
print(conn.execute(query, wanted).fetchall())


SELECT name, latitude FROM stations WHERE name IN (?, ?, ?) ORDER BY latitude DESC
[('Svalbard', 78.22), ('Kirkenes', 69.73), ('Tromso', 69.65)]


The f-string wrote only question marks into the SQL, as many as the program counted, and the names
traveled as parameters, in a list this time, which fills question marks as a tuple does.

### Names that cannot be parameters

SQLite has to know a statement's tables and columns to prepare it, and it prepares the statement
before it binds any value. So a table name, a column name, and a word such as `DESC` cannot be
parameters. `FROM ?` is a syntax error, which Common errors shows, and `ORDER BY ?` is worse: it
runs, and sorts every row by the same value, the text it was given:


In [9]:
latitudes = [latitude for (latitude,) in conn.execute("SELECT latitude FROM stations ORDER BY ?", ("latitude",))]

print("ORDER BY ? sorted the stations:", latitudes == sorted(latitudes))


ORDER BY ? sorted the stations: False


When a user picks the order, the program looks the choice up in a dictionary of the orders it offers,
and writes what it finds there into the SQL. A choice that is not in the dictionary raises `KeyError`
before any SQL is written, so nothing a user types reaches the SQL text:


In [10]:
ORDERS = {"coldest": "r.celsius", "warmest": "r.celsius DESC"}


def readings_at(conn, station, order, limit=3):
    """A station's readings, in an order chosen from ORDERS: the order is looked up, never pasted in."""
    query = f"""
        SELECT r.hour, r.celsius
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE s.name = ? AND r.celsius IS NOT NULL
        ORDER BY {ORDERS[order]}, r.hour
        LIMIT ?
    """
    return conn.execute(query, (station, limit)).fetchall()


print("coldest:", readings_at(conn, "Svalbard", "coldest"))
print("warmest:", readings_at(conn, "Svalbard", "warmest"))
try:
    readings_at(conn, "Svalbard", "r.celsius; DROP TABLE readings")
except KeyError as error:
    print("refused:", error)


coldest: [('2025-01-12T03:00', -17.3), ('2025-01-17T02:00', -17.2), ('2025-01-07T04:00', -17.1)]
warmest: [('2025-07-13T15:00', 8.3), ('2025-07-17T15:00', 8.2), ('2025-07-18T14:00', 8.2)]
refused: 'r.celsius; DROP TABLE readings'


`LIMIT ?` is a placeholder, since the number of rows is a value. The hostile order never reached
the SQL: the lookup failed first, and `readings` is untouched.

### Question marks, names, or an f-string

This notebook has put values into SQL two ways, and names into SQL a third way:

| Write | When | Why |
|---|---|---|
| `?`, filled from a tuple or a list | most statements, with a few values whose order is easy to see | it is short, and it is sqlite3's own style, the `"qmark"` that `sqlite3.paramstyle` names |
| `:name`, filled from a dictionary | many values, a value used twice, or values that arrive as a dictionary | a value is matched to its place by name, not by counting |
| an f-string, with a name taken from a fixed set | a table, a column or a sort order that has to vary | no placeholder can stand for a name, and the fixed set keeps typed text out |

The default is `?`. Change to names once a statement has more values than you can match to their
marks at a glance, and never put a value into SQL with an f-string, wherever the value came from. One
statement uses one style: a dictionary cannot fill `?`, and Python 3.14 refuses a tuple for
`:station`, which Common errors shows.

### A search built from fixed pieces

The pieces of this notebook in one job: a search over the readings that a form might drive, where
every filter is optional. `build_search` writes SQL only from pieces fixed in the function, a
condition for every filter the caller filled in, question marks counted from a list, and an order
looked up in `ORDERS`. Every value the caller gave goes into a list of parameters, in the order of
its question marks. Printing what it builds for a hostile station name shows that the name never
reaches the SQL:


In [11]:
def build_search(stations=None, below=None, above=None, month=None, order="coldest", limit=5):
    """The SQL and the parameters for readings that match every filter given. Only fixed text goes into the SQL."""
    conditions, parameters = ["r.celsius IS NOT NULL"], []
    if stations:
        conditions.append(f"s.name IN ({', '.join('?' * len(stations))})")
        parameters.extend(stations)
    if below is not None:
        conditions.append("r.celsius < ?")
        parameters.append(below)
    if above is not None:
        conditions.append("r.celsius > ?")
        parameters.append(above)
    if month is not None:
        conditions.append("substr(r.hour, 1, 7) = ?")
        parameters.append(month)
    query = f"""
        SELECT s.name, r.hour, r.celsius
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE {' AND '.join(conditions)}
        ORDER BY {ORDERS[order]}, r.hour, s.name
        LIMIT ?"""
    return query, [*parameters, limit]


query, parameters = build_search(stations=["Oslo", "Tromso"], below=-4, month="2025-02", order="warmest", limit=4)
print(conn.execute(query, parameters).fetchall())

query, parameters = build_search(above=15, month="2025-07", limit=3)
print(conn.execute(query, parameters).fetchall())

query, parameters = build_search(stations=["Oslo' OR '1'='1"], below=-4)
print(query)
print(parameters)
print(conn.execute(query, parameters).fetchall())


[('Tromso', '2025-02-01T20:00', -4.1), ('Oslo', '2025-02-04T05:00', -4.1), ('Oslo', '2025-02-04T07:00', -4.1), ('Tromso', '2025-02-04T10:00', -4.1)]
[('Oslo', '2025-07-01T08:00', 15.1), ('Tromso', '2025-07-01T13:00', 15.1), ('Bergen', '2025-07-02T00:00', 15.1)]

        SELECT s.name, r.hour, r.celsius
        FROM readings AS r JOIN stations AS s ON s.id = r.station_id
        WHERE r.celsius IS NOT NULL AND s.name IN (?) AND r.celsius < ?
        ORDER BY r.celsius, r.hour, s.name
        LIMIT ?
["Oslo' OR '1'='1", -4, 5]
[]


The first search asked for the mildest readings below -4 at Oslo and Tromso in February, and the
second for the coolest readings above 15 at any station in July, both from the same function. For the
hostile name, the SQL holds nothing but fixed text and question marks, the name waits in the list of
parameters, and no station is called `Oslo' OR '1'='1`, so nothing matched.

### Where each part came from

| In the search | What it relies on | The section that showed it |
|---|---|---|
| `r.celsius < ?` and `substr(r.hour, 1, 7) = ?` | a placeholder wherever a value goes | A value in its place |
| `parameters`, a list in the order of the marks | question marks filled in order, from a list as well as a tuple | A list of values for IN |
| `s.name IN (?, ?)` written from `len(stations)` | one mark for every value, counted, never copied from the values | A list of values for IN |
| `ORDERS[order]` in the f-string | a name looked up in a fixed set, since no placeholder takes a name | Names that cannot be parameters |
| `?` throughout, with no names | the default style for a statement whose marks are easy to match | Question marks, names, or an f-string |
| `Oslo' OR '1'='1` matching nothing | text through a placeholder stays text | Text that SQL would misread |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/04-parameters-solutions.ipynb).

**1.** Put the name `Svalbard` in a variable, and print that station's latitude, passing the name
through a placeholder.


In [12]:
# your code here


**2.** Add a note for Bergen on `2025-08-19` that reads `Gauge's lid replaced; "no leaks" -- so far`,
then print every note about Bergen, with its day.


In [13]:
# your code here


**3.** Write one statement with the named placeholders `:station`, `:low` and `:high` that counts a
station's hours between two temperatures, and run it for Tromso between -2 and 2, and for Bergen
between 18 and 21.


In [14]:
# your code here


**4.** Put the names `Bergen`, `Oslo` and `Tromso` in a list, and print every one of those stations'
coldest reading, with `IN` and one placeholder for every name in the list.


In [15]:
# your code here


**5.** Write `extreme(conn, station, which)`, where `which` is `"coldest"` or `"warmest"`, choosing
`MIN` or `MAX` from a dictionary, and print both for Oslo. Then show what `extreme` does when `which`
is `"average"`.


In [16]:
# your code here


**6.** Write `count_equal(conn, celsius)`, which counts the readings equal to `celsius`, where
`celsius` may be `None`, and print it for `None` and for `20.8`.


In [17]:
# your code here


## Common errors

### sqlite3.ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 6 supplied.


In [18]:
station = "Bergen"
conn.execute("SELECT latitude FROM stations WHERE name = ?", (station))


ProgrammingError: Incorrect number of bindings supplied. The current statement uses 1, and there are 6 supplied.

`(station)` is not a tuple. Parentheses around a single value only group it, so `execute` was
handed the string itself, and a string is a sequence too, of its characters: six of them for
`Bergen`, for a statement with one placeholder. The comma makes the tuple, and a list of one works
as well:


In [19]:
print(conn.execute("SELECT latitude FROM stations WHERE name = ?", (station,)).fetchone())
print(conn.execute("SELECT latitude FROM stations WHERE name = ?", [station]).fetchone())


(60.39,)
(60.39,)


### sqlite3.OperationalError: near "?": syntax error


In [20]:
table = "readings"
conn.execute("SELECT COUNT(*) FROM ?", (table,))


OperationalError: near "?": syntax error

SQLite reads the structure of a statement, its tables included, before any value is bound, and a
`?` where a table name belongs is not a table name, so the statement does not parse. A table that has
to vary is picked from the tables the program knows, and only then written into the SQL:


In [21]:
TABLES = {"stations", "readings", "notes"}


def count_rows(conn, table):
    """The number of rows in one of TABLES. Any other name is refused before it reaches the SQL."""
    if table not in TABLES:
        raise ValueError(f"no table called {table!r} is offered")
    return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]


print({table: count_rows(conn, table) for table in sorted(TABLES)})


{'notes': 5, 'readings': 35040, 'stations': 5}


### sqlite3.ProgrammingError: You did not supply a value for binding parameter :high.


In [22]:
between = """
    SELECT COUNT(*)
    FROM readings AS r JOIN stations AS s ON s.id = r.station_id
    WHERE s.name = :station AND r.celsius BETWEEN :low AND :high
"""
conn.execute(between, {"station": "Oslo", "low": -1, "hihg": 1})


ProgrammingError: You did not supply a value for binding parameter :high.

The dictionary has no key called `high`, so sqlite3 has nothing to put in `:high`. The misspelled
`hihg` raised nothing by itself, since keys a statement does not use are ignored, and the message
names the placeholder that went unfilled, which points at the key to check:


In [23]:
print(conn.execute(between, {"station": "Oslo", "low": -1, "high": 1}).fetchone()[0])


1017


### sqlite3.ProgrammingError: Error binding parameter 1: type 'list' is not supported


In [24]:
wanted = ["Oslo", "Bergen"]
conn.execute("SELECT name, latitude FROM stations WHERE name IN (?)", (wanted,))


ProgrammingError: Error binding parameter 1: type 'list' is not supported

One `?` takes one value, and a list is not a value SQLite can store, so binding the first parameter
failed. `IN` needs a mark for every value in the list, written from its length:


In [25]:
query = f"SELECT name, latitude FROM stations WHERE name IN ({', '.join('?' * len(wanted))}) ORDER BY name"

print(conn.execute(query, wanted).fetchall())


[('Bergen', 60.39), ('Oslo', 59.91)]


### sqlite3.ProgrammingError: Binding 1 (':station') is a named parameter, but you supplied a sequence which requires nameless (qmark) placeholders.


In [26]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        print("ran:", conn.execute("SELECT latitude FROM stations WHERE name = :station", ("Oslo",)).fetchone())
    except sqlite3.ProgrammingError as error:
        print("ProgrammingError:", error)
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")


ProgrammingError: Binding 1 (':station') is a named parameter, but you supplied a sequence which requires nameless (qmark) placeholders.


A tuple fills question marks, and `:station` is a name. Python 3.14 refuses the mismatch, as here.
Python 3.12 and 3.13 run the statement anyway, filling the names in order, and emit a
`DeprecationWarning` saying that 3.14 will raise. Python shows that warning only when the code that
triggers it is in the script being run, not in a module it imports, so code that ran for years can
stop at an upgrade with no warning ever seen. The cell catches both, whichever version runs it. Give
names a dictionary, or give a tuple question marks:


In [27]:
print(conn.execute("SELECT latitude FROM stations WHERE name = :station", {"station": "Oslo"}).fetchone())
print(conn.execute("SELECT latitude FROM stations WHERE name = ?", ("Oslo",)).fetchone())


(59.91,)
(59.91,)


### sqlite3.OperationalError: near "s": syntax error


In [28]:
note = "The gauge's lid blew off in the storm"
conn.execute(f"INSERT INTO notes (station_id, day, note) VALUES ({station_ids['Bergen']}, '2025-09-30', '{note}')")


OperationalError: near "s": syntax error

The f-string pasted the note between single quotes, and the apostrophe in `gauge's` ended the SQL
text early. SQLite read `'The gauge'` as the whole note, and then met `s` where it expected a comma
or a closing parenthesis. SQL's own escape is to double the apostrophe, `gauge''s`, but code that
escapes by hand has to remember it for every value it ever pastes. Passed as a parameter, the note
needs nothing:


In [29]:
conn.execute("INSERT INTO notes (station_id, day, note) VALUES (?, ?, ?)", (station_ids["Bergen"], "2025-09-30", note))
conn.commit()

print(conn.execute("SELECT day, note FROM notes WHERE station_id = ? ORDER BY day", (station_ids["Bergen"],)).fetchall())


[('2025-09-30', "The gauge's lid blew off in the storm"), ('2025-11-03', 'Rain gauge cleared of leaves')]


### No error, and every note in the file: a station name pasted into the SQL with an f-string


In [30]:
def notes_for(conn, station):
    """Every note about a station, with its day. The name is pasted into the SQL, which is the mistake."""
    query = f"""
        SELECT n.day, n.note
        FROM notes AS n JOIN stations AS s ON s.id = n.station_id
        WHERE s.name = '{station}'
    """
    return conn.execute(query).fetchall()


for typed in ["Oslo", "' OR '1'='1", "' UNION SELECT name, sql FROM sqlite_schema --"]:
    print(f"{typed}:")
    for day, note in notes_for(conn, typed):
        print("   ", day, "|", note)


Oslo:
    2025-07-14 | Mast repainted; readings from 09:00 to 11:00 may run warm
' OR '1'='1:
    2025-03-02 | The sensor's heater failed, so the station sent nothing all day
    2025-07-14 | Mast repainted; readings from 09:00 to 11:00 may run warm
    2025-10-30 | A visitor asked "is it -- really -- always this cold?"
    2025-11-03 | Rain gauge cleared of leaves
    2025-12-01 | Heater checked before winter; it's working
    2025-09-30 | The gauge's lid blew off in the storm
' UNION SELECT name, sql FROM sqlite_schema --:
    notes | CREATE TABLE notes (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                        day TEXT NOT NULL, note TEXT NOT NULL)
    readings | CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL)
    stations | CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL)


`notes_for` worked for `Oslo`, and a test with real station names would pass. The second search
closed the quote around the name itself and added `OR '1'='1'`, which is true for every row, so
every note came back. The third closed the quote, added a second `SELECT` with `UNION`, and turned
the f-string's leftover quote into a comment with `--`: the search returned the name of every table
in the file, with the statement that made it, and a search written the same way could read any of
those tables. sqlite3's `execute` runs one statement at a time, so text such as
`'; DROP TABLE readings; --` raises an error instead of deleting a table, but one statement is
enough to read anything the connection can, and code that runs such SQL through `executescript` gets
no protection at all. Nothing raised in any of it. The fix is a placeholder, in the same place:


In [31]:
def notes_for(conn, station):
    """Every note about a station, with its day, and the name passed as a parameter."""
    query = """
        SELECT n.day, n.note
        FROM notes AS n JOIN stations AS s ON s.id = n.station_id
        WHERE s.name = ?
    """
    return conn.execute(query, (station,)).fetchall()


for typed in ["Oslo", "' OR '1'='1", "' UNION SELECT name, sql FROM sqlite_schema --"]:
    print(f"{typed}: {notes_for(conn, typed)}")
conn.close()


Oslo: [('2025-07-14', 'Mast repainted; readings from 09:00 to 11:00 may run warm')]
' OR '1'='1: []
' UNION SELECT name, sql FROM sqlite_schema --: []


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
database in it:


In [32]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A `?` marks the place for a value, and a tuple or a list fills the marks in order. A tuple of one
  needs its comma, as in `(station,)`.
- A `:name` takes its value from a dictionary, so the order of the keys does not matter, a name can
  appear twice, and extra keys are ignored. A statement uses one style or the other.
- SQLite prepares a statement before it binds any value, so an apostrophe, a semicolon or `--` in a
  parameter is only a character, and a parameter can never change what the statement does.
- `None` goes in as `NULL`, which `= ?` never matches and `IS ?` does, and `int`, `float`, `str` and
  `bytes` go in as integer, real, text and blob.
- `IN` takes one `?` for every value, written from the length of the list, never from what is in it.
- A table name, a column name or `DESC` cannot be a parameter, and `ORDER BY ?` sorts by nothing, so
  a name that has to vary is looked up in a fixed set and then written into the SQL.
- A value pasted into SQL with an f-string breaks on an apostrophe, and lets whoever wrote the value
  rewrite the query: `' OR '1'='1` returned every note, and a `UNION` read `sqlite_schema`.


## What is next

The **Row Factories** notebook turns from the values going into a statement to the rows coming out
of one: rows that answer to a column's name, as `row["celsius"]`, and not only to its position.


---

&#8592; **Previous:** [Tables and Queries](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/03-tables-and-queries.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
